# Quality Factor

Quality strategies rank companies by profitability, balance-sheet strength, and durable margins.

Abbreviations used in this notebook:

- **ROIC**: Return on Invested Capital.
- **D/E**: Debt to Equity.
- **EBITDA**: Earnings Before Interest, Taxes, Depreciation, and Amortization.
- **IR**: Information Ratio.

## 1. Intuition

Quality investing favors companies that can compound capital with strong profitability and manageable leverage. These businesses may be more resilient during stress.

## 2. Mathematics

Quality score:

$$
Score = z(ROIC) + z(Gross\ Margin) + z(-Debt/EBITDA)
$$

Equal-weight quality return:

$$
R_{quality,t} = \frac{1}{N}\sum_i R_{i,t}
$$

Where:
- $\text{Score}$ = composite quality score.
- $\text{z(...)}$ = standardized z-score transformation.
- $\text{ROIC}$ = return on invested capital.
- $\text{Gross Margin}$ = gross profit divided by revenue.
- $\text{Debt/EBITDA}$ = leverage ratio.
- $R_quality,t$ = quality portfolio return at time $t$.
- $N$ = number of selected stocks.


## 3. Implementation

We rank the universe by ROIC, gross margin, and low leverage.

In [ ]:
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "GUIDELINES.md").exists())
helper_path = project_root / "05_strategies" / "strategy_utils.py"
spec = importlib.util.spec_from_file_location("strategy_utils", helper_path)
strategy_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(strategy_utils)

plt.style.use("seaborn-v0_8-whitegrid")
prices, returns, fundamentals = strategy_utils.generate_strategy_universe()
benchmark = returns.mean(axis=1)

quality = fundamentals.copy()
quality["quality_score"] = (
    strategy_utils.zscore(quality["roic"], True)
    + strategy_utils.zscore(quality["gross_margin"], True)
    + strategy_utils.zscore(quality["debt_to_ebitda"], False)
)
quality = quality.sort_values("quality_score", ascending=False)
selected = quality.head(6)["ticker"].tolist()
quality_returns = strategy_utils.equal_weight_return(returns, selected)
quality.head(10)

In [ ]:
summary = pd.DataFrame({
    "quality_strategy": strategy_utils.performance_summary(quality_returns, benchmark),
    "benchmark": strategy_utils.performance_summary(benchmark),
})
summary.round(4)

## 4. Visualization

Quality screens should show the ranking and the characteristics behind the ranking.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
quality.head(10).plot(x="ticker", y="quality_score", kind="bar", ax=axes[0], color="#2f6f8f", legend=False)
axes[0].set_title("Top Quality Scores")
axes[0].tick_params(axis="x", rotation=35)

(1 + pd.DataFrame({"Quality": quality_returns, "Benchmark": benchmark})).cumprod().plot(ax=axes[1], color=["#2f6f8f", "#9a6b2f"])
axes[1].set_title("Quality Strategy vs Benchmark")
axes[1].set_ylabel("Growth of 1")
plt.tight_layout(); plt.show()

## 5. Application

Quality can be used as a standalone strategy or as a filter for value and momentum. It helps avoid companies that look cheap or strong only because of temporary or fragile conditions.

In [ ]:
quality[quality["ticker"].isin(selected)].set_index("ticker")[["roic", "gross_margin", "debt_to_ebitda", "pe"]].round(3)

## 6. Reflection

- Quality focuses on business strength.
- High quality can be expensive, so valuation still matters.
- Balance-sheet strength can reduce downside risk.
- Accounting definitions must be consistent across companies.

Questions to answer after running the notebook:

1. Which metrics drive the quality ranking?
2. Did quality outperform the benchmark?
3. Are the selected stocks expensive?
4. How would you combine quality with value?